# 🎭 Teddy Face Swap — GPU (free test)

**Just 2 taps, then wait ~4 minutes:**

1. Top menu → **Runtime → Change runtime type → T4 GPU → Save**  *(may already be set)*
2. Top menu → **Runtime → Run all**

When it finishes, a **https link** appears at the very bottom of the last cell. **Tap that link on your phone** → *Start Camera* → smooth face swap on a real GPU.

⚠️ Keep this Colab tab open while you use it — if it closes, the server stops (that's the free-tier limit).

In [ ]:
# 1/7  Check we actually got a GPU
import subprocess
o = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
if o.returncode or not o.stdout.strip():
    raise SystemExit('NO GPU. Fix: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.')
print('GPU:', o.stdout.strip())

In [ ]:
# 2/7  Get the app code
import os
REPO='https://github.com/oluwacoded/Deepfakcall'; BRANCH='main'
if not os.path.isdir('/content/Deepfakcall'):
    !git clone --depth 1 -b {BRANCH} {REPO} /content/Deepfakcall
os.chdir('/content/Deepfakcall'); print('Now in', os.getcwd()); print('has web_server.py:', os.path.exists('web_server.py'), '| has templates:', os.path.exists('templates/index.html'))

In [ ]:
# 3/7  Install dependencies (GPU build of onnxruntime) + tunnel tool
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1
!pip -q install -r requirements-gpu.txt
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared
print('dependencies installed')

In [ ]:
# 4/7  Confirm the GPU is usable by the AI runtime
import onnxruntime as ort
p = ort.get_available_providers(); print('providers:', p)
print('GPU ready' if 'CUDAExecutionProvider' in p else 'WARNING: no CUDA provider - copy this output to the agent')

In [ ]:
# 5/7  Pre-download the Amber Song face model (~718 MB) straight to this GPU box
import os; os.makedirs('userdata/models',exist_ok=True)
m='userdata/models/Amber_Song.dfm'
if not os.path.exists(m) or os.path.getsize(m)<700_000_000:
    !wget -q --show-progress -O {m} https://github.com/iperov/DeepFaceLive/releases/download/AMBER_SONG/Amber_Song.dfm
!ls -lh userdata/models/

In [ ]:
# 6/7  Start the app + a public link
import subprocess, time, re, os, sys
os.environ['SESSION_SECRET']='colab-'+os.urandom(8).hex()
srv=subprocess.Popen([sys.executable,'web_server.py'],stdout=open('server.log','w'),stderr=subprocess.STDOUT)
print('starting server...'); time.sleep(10)
tun=subprocess.Popen(['/tmp/cloudflared','tunnel','--url','http://localhost:5000','--no-autoupdate'],stdout=open('tunnel.log','w'),stderr=subprocess.STDOUT)
url=None
for _ in range(45):
    time.sleep(2)
    log=open('tunnel.log').read() if os.path.exists('tunnel.log') else ''
    mm=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com',log)
    if mm: url=mm.group(0); break
print('\n'+'='*58)
if url:
    print('OPEN THIS ON YOUR PHONE (tap it):\n\n    '+url+'\n')
    print('  1) tap Start Camera, allow the camera')
    print('  2) Amber Song is preloaded - tap Load Model')
    print('  3) keep this Colab tab open')
else:
    print('Tunnel not ready - run the next cell and send the agent the output.')
print('='*58)

In [ ]:
# 7/7  Keeps the session awake + shows live activity. Leave this running.
import time
while True:
    time.sleep(30)
    try: print(''.join(open('server.log').readlines()[-6:]), flush=True)
    except Exception as e: print('log:',e)